<a href="https://colab.research.google.com/github/ayonya100/fawkes/blob/collab-support/colab-demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Google Colab with GitHub



In [1]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [ ]:
from os.path import join  

ROOT = '/content/gdrive'
# path to your project on Google Drive
MY_GOOGLE_DRIVE_PATH = 'My Drive/fawkes-forked' 
# replace with your Github username 
GIT_USERNAME = "ayonya100" 
# definitely replace with your
GIT_TOKEN = ""  
# Replace with your github repository in this case we want 
# to clone deep-learning-v2-pytorch repository
GIT_REPOSITORY = "fawkes" 

PROJECT_PATH = join(ROOT, MY_GOOGLE_DRIVE_PATH)

# It's good to print out the value if you are not sure 
print("PROJECT_PATH: ", PROJECT_PATH)   

# In case we haven't created the folder already; we will create a folder in the project path 
!mkdir "{PROJECT_PATH}"    

#GIT_PATH = "https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPOSITORY}.git" this return 400 Bad Request for me
GIT_PATH = "https://" + GIT_TOKEN + "@github.com/" + GIT_USERNAME + "/" + GIT_REPOSITORY + ".git"
print("GIT_PATH: ", GIT_PATH)

In [2]:
cd 

/root


In [3]:
print("PROJECT_PATH: ", PROJECT_PATH)   


NameError: ignored

In [ ]:
!git clone "{GIT_PATH}"

Cloning into 'fawkes'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (240/240), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 240 (delta 97), reused 186 (delta 59), pack-reused 0
Receiving objects: 100% (240/240), 271.85 KiB | 4.85 MiB/s, done.
Resolving deltas: 100% (97/97), done.


In [6]:
ls -a

./   .bash_history  .cache/   .gsutil/   .jupyter/  .local/     .npm/
../  .bashrc        .config/  .ipython/  .keras/    .node-gyp/  .profile


In [ ]:
!pip3 install -r requirements.txt

     |████████████████████████████████| 71kB 1.9MB/s 
     |████████████████████████████████| 1.6MB 4.3MB/s 
     |████████████████████████████████| 81kB 5.8MB/s 
     |████████████████████████████████| 194kB 12.7MB/s 
     |████████████████████████████████| 174kB 13.0MB/s 
     |████████████████████████████████| 276kB 18.0MB/s 
     |████████████████████████████████| 4.9MB 16.5MB/s 
     |████████████████████████████████| 102kB 9.9MB/s 
     |████████████████████████████████| 337kB 43.0MB/s 
  Created wheel for pyLDAvis: filename=pyLDAvis-2.1.2-py2.py3-none-any.whl size=97712 sha256=f95fcf09f44ae4514f480b320541b39938e47f2ba9ff93b9478823254e9561db
  Stored in directory: /root/.cache/pip/wheels/98/71/24/513a99e58bb6b8465bae4d2d5e9dba8f0bef8179e3051ac414
  Created wheel for python-http-client: filename=python_http_client-3.3.1-cp36-none-any.whl size=7523 sha256=d408993296da94ccff54471add9ef236621945ae74f2f873dc421cc705dfec13
  Stored in directory: /root/.cache/pip/wheels/14/67/9b/abeb3c1

In [75]:
cd /content/gdrive/My\ Drive/fawkes-forked/fawkes

/content/gdrive/My Drive/fawkes-forked/fawkes


In [77]:
import re
import sys
import os
import json

import nltk
nltk.download('stopwords')
nltk.download('wordnet')


from nltk.stem.wordnet import WordNetLemmatizer

sys.path.append(os.path.realpath("."))

from src.utils import *
from src.config import *

lmtzr = WordNetLemmatizer()


def parse_keywords_file(keyword_file_name, enable_remove_stop_words=True):
    # Topics is a dict, key = Topic Name. value = list of words and weights.
    topics = {}
    keywords_list = open_json(keyword_file_name)
    for topic_keyword in keywords_list:
        topic = {}
        line = " ".join(keywords_list[topic_keyword])

        # Remove all trailing and beginning write spaces
        line = line.lower()
        line = line.strip()
        # We will replace all the non-alphabet charectors with a space
        cleaned_line = re.sub("[^a-zA-Z]+", " ", line)
        # Replace multiple spaces with a single space
        cleaned_line = re.sub(" +", " ", cleaned_line)
        # Split the line according to space to get the words
        cleaned_line = cleaned_line.split()
        # Remove the stopwords.
        if enable_remove_stop_words:
            cleaned_line = remove_stop_words(cleaned_line)
        # For each word assign a weight
        for word in list(set(cleaned_line)):
            # Add the word to the topic
            topic[lmtzr.lemmatize(word.lower())] = 1
        topics[topic_keyword] = topic
    return topics


if __name__ == "__main__":
    app_configs = open_json(
        APP_CONFIG_FILE.format(file_name=APP_CONFIG_FILE_NAME))

    for app in app_configs:
        app_config = open_json(APP_CONFIG_FILE.format(file_name=app))
        if validate_app_config(app_config):
            app_config = decrypt_config(app_config)
            dump_json(parse_keywords_file(app_config[TOPIC_KEYWORDS_FILE]),
                      TOPICS_WEIGHT_FILE.format(app=app_config[APP]))
            dump_json(parse_keywords_file(app_config[BUG_FEATURE_FILE], False),
                      BUG_FEATURE_FILE_WITH_WEIGHTS.format(app=app_config[APP]))


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[LOG] Validating app-config schema
[LOG] Validating app-config schema successful
